In [3]:
"""
ШЧ-2: очистка, временные ряды, кластеризация, корреляции, SPC (Шухарт/EWMA/CUSUM).

Зависимости:
    pip install polars openpyxl scikit-learn scipy matplotlib numpy

Запуск:
    python shch2_pipeline.py --input "ШЧ_2_corr.xlsx" --outdir out
"""

from __future__ import annotations

import argparse
import math
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.preprocessing import StandardScaler
import openpyxl
warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)


polars.config.Config

In [7]:
# ----------------------------------------------------------------------
# Константы: имена колонок исходного файла
# ----------------------------------------------------------------------------
C_DATE      = "Дата осмотра"
C_INSPECTOR = "Осмотр проводил"
C_STATION   = "Станция, перегон"
C_OBJ       = "Объекты, количество отступлений"
C_GROUP     = "Классификация (Группа)"
C_KIND      = "Классификация (Вид)"
C_NOTE      = "Примечание"
C_DEADLINE  = "Крайний срок"
C_STATUS    = "Статус"
C_REASON    = "Причина"
C_FIXED     = "Дата устранения"

TEST_TOKENS = {"тест", "test", "-", "dddd", "ккквак", "ттт", "."}

# 1. ЗАГРУЗКА
def load_all_sheets(path) -> pl.DataFrame:
    """Читает все листы Excel и склеивает в один DataFrame с колонкой sheet."""   

    wb = openpyxl.load_workbook(path, read_only=True) # через библиотеку получаем список всех листов
    sheets = wb.sheetnames # список наших листов
    wb.close() # завершанием чтение файла 

    frames = [] #обзявляем пустов список, в который будем записывать считанные листы
    for sh in sheets: # в цикле считываем данные с листов c попыткой считать и выводом информации об ошике, если не сложилось
        try: 
            d = pl.read_excel(path, sheet_name=sh)
        except Exception as e:  # noqa: BLE001
            print(f"  [!] лист '{sh}' не прочитан: {e}")
            continue
        if d.height == 0:
            continue
        # унифицируем: всё в строки, чтобы vstack не падал на разных типах
        d = d.select([pl.col(c).cast(pl.Utf8, strict=False) for c in d.columns])
        d = d.with_columns(pl.lit(sh).alias("sheet"))
        frames.append(d) #пополянем ранее сформированный список 
        print(f"  лист '{sh}': {d.height} строк, {d.width} колонок")

    if not frames:
        raise RuntimeError("Не удалось прочитать ни один лист")

    # выравниваем набор колонок
    all_cols = []
    for f in frames:
        for c in f.columns:
            if c not in all_cols:
                all_cols.append(c)
    frames = [
        f.with_columns([pl.lit(None, dtype=pl.Utf8).alias(c)
                        for c in all_cols if c not in f.columns]).select(all_cols)
        for f in frames
    ]
    df = pl.concat(frames, how="vertical") #объединияем все данные из списка датафреймов в один массив 
    # print(f"  ИТОГО: {df.height} строк")
    return df

Path = '../лекция 1/ШЧ_2_corr.xlsx'
df = load_all_sheets(Path)
df[:3]

Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string


  лист '2019': 12112 строк, 13 колонок
  лист '2018': 11110 строк, 13 колонок


Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string


  лист '2017': 12052 строк, 13 колонок
  лист '2016': 2602 строк, 13 колонок
  лист '2015': 4684 строк, 13 колонок


Could not determine dtype for column 10, falling back to string


  лист '2014': 3431 строк, 13 колонок


__UNNAMED__0,Дата осмотра,Осмотр проводил,"Станция, перегон","Объекты, количество отступлений",Классификация (Группа),Классификация (Вид),Примечание,Крайний срок,Статус,Причина,Дата устранения,sheet
str,str,str,str,str,str,str,str,str,str,str,str,str
"""0""","""2019-04-03""","""Сотрудник_0""","""станция_0""","""327""","""группа_0""","""вид_0""","""Стрелка №39 устранить люфт дли…","""2019-04-13""","""ЗАКРЫТО""",null,"""2019-04-11""","""2019"""
"""1""","""2019-04-03""","""Сотрудник_0""","""станция_0""","""327""","""группа_0""","""вид_0""","""Стрелка №25 отрегулировать раб…","""2019-04-13""","""ЗАКРЫТО""",null,"""2019-04-11""","""2019"""
"""2""","""2019-04-03""","""Сотрудник_0""","""станция_0""","""327""","""группа_0""","""вид_0""","""Стрелка №31 устранить люфт дли…","""2019-04-13""","""ЗАКРЫТО""",null,"""2019-04-11""","""2019"""


In [6]:
df['sheet'].value_counts()

sheet,count
str,u32
"""2019""",12112
"""2015""",4684
"""2018""",11110
"""2014""",3431
"""2017""",12052
"""2016""",2602


In [ ]:
# 2. ОЧИСТКА
def parse_date_col(name: str, alias: str) -> pl.Expr:
    """Робастный парсинг даты: пробуем несколько форматов."""
    c = pl.col(name).cast(pl.Utf8, strict=False).str.strip_chars()
    return (
        pl.coalesce([
            c.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
            c.str.to_datetime("%Y-%m-%d", strict=False),
            c.str.to_datetime("%d.%m.%Y", strict=False),
        ])
        .dt.date()
        .alias(alias)
    )


def clean(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Возвращает (clean_df, report_df).
    Ничего не удаляет молча: проблемные строки помечаются флагами.
    """
    log: list[dict] = []

    def note(step: str, value):
        log.append({"step": step, "value": str(value)})

    n0 = df.height
    note("исходных строк", n0)

    # --- 2.1 нормализация текста ---
    text_cols = [c for c in (C_INSPECTOR, C_STATION, C_GROUP, C_KIND,
                             C_STATUS, C_NOTE, C_REASON) if c in df.columns]
    df = df.with_columns([
        pl.col(c).cast(pl.Utf8, strict=False)
          .str.replace_all(r"[\r\n]+", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          .alias(c)
        for c in text_cols
    ])
    df = df.with_columns([
        pl.when(pl.col(c).str.len_chars() == 0).then(None).otherwise(pl.col(c)).alias(c)
        for c in text_cols
    ])

    # --- 2.2 даты ---
    df = df.with_columns([
        parse_date_col(C_DATE,     "d_insp"),
        parse_date_col(C_DEADLINE, "d_dead"),
        parse_date_col(C_FIXED,    "d_fix"),
    ])
    note("null d_insp после парсинга", df["d_insp"].null_count())
    note("null d_dead после парсинга", df["d_dead"].null_count())
    note("null d_fix  после парсинга", df["d_fix"].null_count())

    # --- 2.3 расщепление смешанной колонки «Объекты, количество отступлений» ---
    if C_OBJ in df.columns:
        raw = pl.col(C_OBJ).cast(pl.Utf8, strict=False).str.strip_chars()
        df = df.with_columns([
            raw.str.extract(r"^(\d+)$", 1).cast(pl.Int64, strict=False).alias("obj_count"),
            raw.alias("obj_raw"),
        ])
        df = df.with_columns(
            pl.when(pl.col("obj_count").is_null())
              .then(pl.col("obj_raw"))
              .otherwise(None)
              .alias("obj_label")
        )
        note("числовых obj_count", df["obj_count"].is_not_null().sum())
        note("текстовых obj_label", df["obj_label"].is_not_null().sum())

    # --- 2.4 статусы ---
    if C_STATUS in df.columns:
        df = df.with_columns(
            pl.col(C_STATUS).str.to_uppercase().alias("status_norm")
        )
        note("уникальных статусов", df["status_norm"].unique().to_list())

    # --- 2.5 производные признаки-интервалы ---
    df = df.with_columns([
        (pl.col("d_dead") - pl.col("d_insp")).dt.total_days().alias("norm_days"),
        (pl.col("d_fix")  - pl.col("d_insp")).dt.total_days().alias("lead_days"),
        (pl.col("d_fix")  - pl.col("d_dead")).dt.total_days().alias("overdue_days"),
    ])

    # --- 2.6 флаги качества ---
    is_test = pl.col(C_NOTE).str.to_lowercase().is_in(list(TEST_TOKENS))
    df = df.with_columns([
        pl.col("d_insp").is_null().alias("flag_no_insp_date"),
        (pl.col("lead_days") < 0).fill_null(False).alias("flag_fix_before_insp"),
        (pl.col("norm_days") < 0).fill_null(False).alias("flag_dead_before_insp"),
        (pl.col("lead_days") > 365 * 3).fill_null(False).alias("flag_lead_gt_3y"),
        is_test.fill_null(False).alias("flag_test_note"),
        ((pl.col("status_norm") != "НОВОЕ") & pl.col("d_fix").is_null())
            .fill_null(False).alias("flag_closed_no_fixdate"),
        (pl.col("d_fix") == pl.date(2019, 1, 26)).fill_null(False)
            .alias("flag_mass_close_20190126"),
    ])
    for f in [c for c in df.columns if c.startswith("flag_")]:
        note(f, df[f].sum())

    # --- 2.7 дубликаты ---
    note("полных дублей", df.is_duplicated().sum())
    bkey = [c for c in ("d_insp", C_STATION, "obj_raw", C_GROUP, C_KIND)
            if c in df.columns]
    n_before = df.height
    df = df.unique(subset=bkey, keep="first", maintain_order=True)
    note("удалено дублей по бизнес-ключу", n_before - df.height)

    # --- 2.8 выбросы lead_days: robust z / IQR ---
    ld = df.filter(pl.col("lead_days").is_not_null())["lead_days"].to_numpy()
    if ld.size > 10:
        q1, q3 = np.percentile(ld, [25, 75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        med = np.median(ld)
        mad = np.median(np.abs(ld - med)) or 1.0
        note("lead_days IQR-границы", (round(lo, 1), round(hi, 1)))
        note("lead_days вне IQR", int(((ld < lo) | (ld > hi)).sum()))
        df = df.with_columns([
            ((pl.col("lead_days") < lo) | (pl.col("lead_days") > hi))
                .fill_null(False).alias("flag_lead_outlier_iqr"),
            (0.6745 * (pl.col("lead_days") - med) / mad).abs()
                .alias("lead_robust_z"),
        ])
        # winsorize для моделирования (исходное значение сохраняется)
        df = df.with_columns(
            pl.col("lead_days").clip(0, max(hi, 1)).alias("lead_days_w")
        )

    # --- 2.9 «валидное ядро» для анализа ---
    df = df.with_columns(
        (~pl.col("flag_no_insp_date")
         & ~pl.col("flag_test_note")
         & ~pl.col("flag_fix_before_insp")).alias("is_valid")
    )
    note("валидных строк (is_valid)", df["is_valid"].sum())

    return df, pl.DataFrame(log)


# ============================================================================
# 3. ВРЕМЕННЫЕ РЯДЫ
# ============================================
def calendar_features(df: pl.DataFrame, date_col: str = "d_insp") -> pl.DataFrame:
    """Календарные признаки + циклическое кодирование."""
    d = pl.col(date_col)
    return df.with_columns([
        d.dt.year().alias("year"),
        d.dt.quarter().alias("quarter"),
        d.dt.month().alias("month"),
        d.dt.week().alias("iso_week"),
        d.dt.day().alias("day"),
        d.dt.weekday().alias("weekday"),
        d.dt.ordinal_day().alias("doy"),
        (d.dt.weekday() >= 6).alias("is_weekend"),
        d.dt.month_start().alias("month_start"),
        # циклические
        (2 * math.pi * d.dt.month() / 12).sin().alias("month_sin"),
        (2 * math.pi * d.dt.month() / 12).cos().alias("month_cos"),
        (2 * math.pi * d.dt.weekday() / 7).sin().alias("dow_sin"),
        (2 * math.pi * d.dt.weekday() / 7).cos().alias("dow_cos"),
    ])


def build_series(df: pl.DataFrame, every: str, date_col: str = "d_insp") -> pl.DataFrame:
    """
    Строит регулярный временной ряд с заданной дискретностью.
    every: '1d' | '1w' | '1mo' | '1q' | '1y'
    """
    base = (
        df.filter(pl.col("is_valid"))
          .select([
              pl.col(date_col).dt.truncate(every).alias("t"),
              pl.col(C_STATION).alias("station"),
              pl.col(C_INSPECTOR).alias("inspector"),
              pl.col(C_GROUP).alias("grp"),
              pl.col("obj_count"),
              pl.col("lead_days_w"),
              pl.col("overdue_days"),
          ])
          .group_by("t")
          .agg([
              pl.len().alias("n"),
              pl.col("station").n_unique().alias("n_stations"),
              pl.col("inspector").n_unique().alias("n_inspectors"),
              pl.col("grp").n_unique().alias("n_groups"),
              pl.col("obj_count").sum().alias("obj_sum"),
              pl.col("lead_days_w").mean().alias("lead_mean"),
              pl.col("lead_days_w").median().alias("lead_median"),
              (pl.col("overdue_days") > 0).mean().alias("overdue_rate"),
          ])
          .sort("t")
    )
    # регулярная сетка + нули
    ser = base.upsample(time_column="t", every=every).with_columns([
        pl.col("n").fill_null(0),
        pl.col("n_stations").fill_null(0),
        pl.col("n_inspectors").fill_null(0),
        pl.col("n_groups").fill_null(0),
        pl.col("obj_sum").fill_null(0),
    ])
    return ser
